In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("")), "src"))

import matplotlib
matplotlib.use("Agg")  # headless backend — no GUI required

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from yield_curve import YieldCurve, NelsonSiegel, PortfolioAnalyzer, fetch_prices_from_yfinance

sns.set_theme(style="whitegrid", palette="dark")
np.set_printoptions(suppress=True, formatter={'float_kind': '{:0.4f}'.format})

OUT = os.path.dirname(os.path.abspath(""))  # repo root when run from there
print(f"Working directory: {os.getcwd()}")
print(f"Output dir for plots: {OUT}")

Working directory: C:\Users\HP\Documents\Bootstrap-Yield-Curve\notebooks
Output dir for plots: C:\Users\HP\Documents\Bootstrap-Yield-Curve


# Bootstrap Yield Curve — Package Demo

A live walkthrough of the `yield_curve` package: bootstrapping Treasury discount factors three ways, fitting a Nelson-Siegel curve, and running a Markowitz portfolio analysis on five large-cap tech stocks.

> **Original exercise:** August 2022 (see `notebooks/Bootstrapping_Yield_Curve.ipynb`).
> **Package surface:** September 2026.
> **Run:** `python notebooks/demo.py` (no Jupyter required) or open this notebook in Jupyter / VS Code.

## 1  Treasury yield curve — bootstrapping three ways

We start with the 10 Treasury bonds from the original exercise. Each row is `(maturity_years, dirty_price, annual_coupon_rate)` at par value 100. The `YieldCurve` class builds the cash-flows matrix and bootstraps discount factors.

In [2]:
# 10 Treasury bonds (maturity, price, coupon) from the original project
maturities = np.arange(1, 11)
prices     = np.array([96.60, 93.71, 91.56, 90.24, 89.74, 90.04, 91.09, 92.82, 95.19, 98.14])
coupons    = np.linspace(0.015, 0.0375, num=10)

bonds = np.column_stack((maturities, prices, coupons))
bonds

array([[1.0000, 96.6000, 0.0150],
       [2.0000, 93.7100, 0.0175],
       [3.0000, 91.5600, 0.0200],
       [4.0000, 90.2400, 0.0225],
       [5.0000, 89.7400, 0.0250],
       [6.0000, 90.0400, 0.0275],
       [7.0000, 91.0900, 0.0300],
       [8.0000, 92.8200, 0.0325],
       [9.0000, 95.1900, 0.0350],
       [10.0000, 98.1400, 0.0375]])

In [3]:
yc = YieldCurve(bonds)

print('Discount factors — Matrix operations:')
print(yc.discount_factors('Matrix operations'))
print()
print('Discount factors — Global Solver:')
print(yc.discount_factors('Global Solver'))
print()
print('Discount factors — Iterative procedure:')
print(yc.discount_factors('Iterative Procedure'))

Discount factors — Matrix operations:
[0.9517 0.9046 0.8612 0.8227 0.7892 0.7604 0.7361 0.7156 0.6985 0.6842]

Discount factors — Global Solver:
[0.9517 0.9046 0.8612 0.8227 0.7892 0.7604 0.7361 0.7156 0.6985 0.6842]

Discount factors — Iterative procedure:
[0.9517 0.9046 0.8612 0.8227 0.7892 0.7604 0.7361 0.7156 0.6985 0.6842]


**Sanity check:** all three methods should give the same discount factors (to within numerical tolerance). The matrix method is exact; the solver and iterative procedure are alternative routes to the same answer.

In [4]:
df_m   = yc.discount_factors('Matrix operations')
df_s   = yc.discount_factors('Global Solver')
df_i   = yc.discount_factors('Iterative Procedure')

print('Max abs diff (matrix vs solver):', np.max(np.abs(df_m - df_s)))
print('Max abs diff (matrix vs iterative):', np.max(np.abs(df_m - df_i)))

# Reproduce market prices from the discount factors
print()
print('Prices recovered from cash flows @ DF (should match market):')
print(yc.cash_flows @ df_m)

Max abs diff (matrix vs solver): 5.84917858592604e-09
Max abs diff (matrix vs iterative): 3.3306690738754696e-16

Prices recovered from cash flows @ DF (should match market):
[96.6000 93.7100 91.5600 90.2400 89.7400 90.0400 91.0900 92.8200 95.1900
 98.1400]


## 2  Spot rates, YTM, and forward rates

From the discount factors we derive the three standard rate views.

In [5]:
spot  = yc.spot_rates()
ytm   = yc.bonds_ytm()
fwd   = yc.forward_rates()

print('Spot rates (%):', np.round(100 * spot, 2))
print('YTM (%):       ', np.round(100 * ytm, 2))
print('1y Forward (%):', np.round(100 * fwd, 2))

Spot rates (%): [5.0700 5.1400 5.1100 5.0000 4.8500 4.6700 4.4700 4.2700 4.0700 3.8700]
YTM (%):        [5.0700 5.1400 5.1100 5.0000 4.8600 4.6900 4.5100 4.3300 4.1500 3.9800]
1y Forward (%): [5.2100 5.0400 4.6800 4.2600 3.7800 3.3000 2.8700 2.4500 2.0800]


In [6]:
yc.plot_rates(
    title='US Treasury Yield Curve — Spot, YTM, and Forward Rates',
    save_path=os.path.join(OUT, 'yield_curve_overview.png'),
    show=False
)
print('Saved: yield_curve_overview.png')

Saved: yield_curve_overview.png


## 3  Nelson-Siegel parametric fit

The bootstrapped curve from 10 bonds is jagged. Nelson-Siegel smooths it into a parametric form with four parameters: level $\beta_0$, slope $\beta_1$, curvature $\beta_2$, and decay $\tau$:

$$f(t) = \beta_0 + \beta_1 e^{-t/\tau} + \beta_2 \frac{t}{\tau} e^{-t/\tau}$$

We fit it to the bootstrapped discount factors by least squares.

In [7]:
ns = NelsonSiegel(
    maturities=bonds[:, 0].astype(float),
    discount_factors=yc.discount_factors('Matrix operations'),
)
ns.fit()

beta0, beta1, beta2, tau = ns.params
print('Fitted Nelson-Siegel parameters:')
print(f'  beta0 (level)     = {beta0:.6f}')
print(f'  beta1 (slope)     = {beta1:.6f}')
print(f'  beta2 (curvature) = {beta2:.6f}')
print(f'  tau (decay)       = {tau:.4f}')

Fitted Nelson-Siegel parameters:
  beta0 (level)     = 0.021700
  beta1 (slope)     = 0.016016
  beta2 (curvature) = 0.069573
  tau (decay)       = 2.0113


In [8]:
ns.plot(
    title='Nelson-Siegel fit vs bootstrapped discount factors',
    save_path=os.path.join(OUT, 'nelson_siegel_fit.png'),
    show=False
)
print('Saved: nelson_siegel_fit.png')

Saved: nelson_siegel_fit.png


## 4  Portfolio analysis — five tech stocks

Now the equity side: daily price panel for AAPL, IBM, MSFT, GOOG, AMZN. We try to fetch live data from yfinance; if that fails (no network or cache issue), we fall back to realistic synthetic data so the analysis still runs end-to-end.

> Requires `yfinance` for live data (`pip install yfinance`). The rest of the demo works without it.

In [9]:
tickers = ['AAPL', 'IBM', 'MSFT', 'GOOG', 'AMZN']

prices = None
try:
    prices = fetch_prices_from_yfinance(tickers, start='2019-01-01')
    if prices is None or prices.empty or len(prices) < 20:
        raise ValueError(f"yfinance returned {len(prices) if prices is not None else 0} rows")
    print(f'Fetched {len(prices)} daily observations for {len(tickers)} tickers')
    print('Date range:', prices.index.min().date(), 'to', prices.index.max().date())
except Exception as e:
    print(f'yfinance unavailable or returned no data ({type(e).__name__}: {e})')
    print('Generating realistic synthetic data for the demo...')
    np.random.seed(42)
    dates = pd.bdate_range('2019-01-02', periods=504)  # ~2 years of trading days
    mu = np.array([0.0005, 0.0003, 0.0006, 0.0005, 0.0007])  # ~12-18% annualized
    vols = np.array([0.018, 0.016, 0.017, 0.019, 0.021])  # daily vol
    corr = np.array([
        [1.00, 0.45, 0.60, 0.55, 0.50],
        [0.45, 1.00, 0.50, 0.42, 0.30],
        [0.60, 0.50, 1.00, 0.65, 0.56],
        [0.55, 0.42, 0.65, 1.00, 0.60],
        [0.50, 0.30, 0.56, 0.60, 1.00],
    ])
    L = np.linalg.cholesky(corr)
    Z = np.random.randn(len(dates), 5)
    returns = Z @ L.T * vols + mu / 252
    prices = pd.DataFrame(
        100 * np.exp(np.cumsum(returns, axis=0)),
        index=dates,
        columns=tickers,
    )
    print(f'Generated {len(prices)} synthetic daily prices for {len(tickers)} tickers')


yfinance unavailable or returned no data (KeyError: 'Adj Close')
Generating realistic synthetic data for the demo...
Generated 504 synthetic daily prices for 5 tickers


In [10]:
prices.head()

,AAPL,IBM,MSFT,GOOG,AMZN
2019-01-02,100.898295,100.160323,101.288541,103.066071,101.424719
2019-01-03,100.474156,102.273649,102.758675,103.276787,102.583572
2019-01-04,99.639733,101.257024,102.384482,100.079655,98.336481
2019-01-07,98.636544,99.399094,101.759674,98.092580,95.128808
2019-01-08,101.273571,100.130153,103.280321,97.621536,95.009495


In [11]:
# Build the analyzer — risk-free rate can come from the bootstrapped curve or an external source
risk_free = float(spot[0])  # 1-year spot rate as a proxy, or set manually: risk_free = 0.045
pa = PortfolioAnalyzer(prices, risk_free_rate=risk_free)

print(f'Risk-free rate used: {risk_free:.4f} ({100*risk_free:.2f}%)')
print(f'Observation period: {len(pa.returns)} trading days')
print(f'Annualized mean returns:')
print(pa.mean_returns.sort_values(ascending=False))

Risk-free rate used: 0.0507 (5.07%)
Observation period: 503 trading days
Annualized mean returns:
AMZN    0.554540
AAPL    0.252183
MSFT    0.233280
GOOG    0.224073
IBM     0.135575
dtype: float64


In [12]:
pa.cumulative_returns_plot(
    title=f'Cumulative returns ({prices.index.min().date()} to {prices.index.max().date()})',
    save_path=os.path.join(OUT, 'cumulative_returns.png'),
    show=False
)
print('Saved: cumulative_returns.png')

Saved: cumulative_returns.png


In [13]:
pa.correlation_heatmap(
    save_path=os.path.join(OUT, 'correlation_heatmap.png'),
    show=False
)
print('Saved: correlation_heatmap.png')

Saved: correlation_heatmap.png


In [14]:
frontier = pa.efficient_frontier(n_points=100)
max_sharpe = pa.max_sharpe_portfolio()

print('Maximum Sharpe portfolio:')
for ticker, w in zip(tickers, max_sharpe['weights']):
    if w > 1e-6:
        print(f'  {ticker:6s}: {w*100:6.2f}%')
print(f'  Volatility : {max_sharpe["volatility"]*100:.2f}%')
print(f'  Return     : {max_sharpe["return"]*100:.2f}%')
print(f'  Sharpe     : {max_sharpe["sharpe"]:.3f}')

Maximum Sharpe portfolio:
  AMZN  : 100.00%
  Volatility : 32.92%
  Return     : 55.45%
  Sharpe     : 1.530


In [15]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(frontier['volatility']*100, frontier['return']*100, 'b-', alpha=0.5, label='Efficient frontier')
ax.scatter(
    [max_sharpe['volatility']*100],
    [max_sharpe['return']*100],
    color='red', s=150, zorder=5, label=f'Max Sharpe ({max_sharpe["sharpe"]:.2f})'
)
for ticker, ret, vol in zip(tickers, pa.mean_returns, np.sqrt(np.diag(pa.cov_matrix))):
    ax.scatter(vol*100, ret*100, alpha=0.6, label=ticker)

ax.set_xlabel('Annualized volatility (%)')
ax.set_ylabel('Annualized return (%)')
ax.set_title('Markowitz Efficient Frontier — Tech Portfolio')
ax.legend()
ax.grid(alpha=0.3)
fig.savefig(os.path.join(OUT, 'efficient_frontier.png'), dpi=150, bbox_inches='tight')
print('Saved: efficient_frontier.png')
plt.close('all')

Saved: efficient_frontier.png


## 5  What this repo demonstrates

A hiring manager looking at this repo should see:

- **Quant finance fundamentals:** bootstrapping, spot/forward rates, Nelson-Siegel — all implemented, not just called from a library.
- **Python package hygiene:** `src/` layout, `pyproject.toml`, type-aware code, `pytest` suite, GitHub Actions CI on Python 3.9–3.12.
- **Data pipeline:** fetching from an external API (yfinance), cleaning, analyzing, visualizing, saving artifacts.
- **Portfolio theory:** mean-variance optimization, efficient frontier, max-Sharpe tangency portfolio — the bread-and-butter of quant / data-science finance roles.
- **Original work preserved:** the 2022 notebook is untouched in `notebooks/`, showing the evolution from exploration to engineered package.

### Outputs produced by this notebook

Running this notebook produces five PNG files in the repo root:

| File | Contents |
|------|----------|
| `yield_curve_overview.png` | Spot rates, YTM, and forward rates on one chart |
| `nelson_siegel_fit.png` | Market vs fitted discount factors |
| `cumulative_returns.png` | Cumulative returns for all five tickers |
| `correlation_heatmap.png` | Correlation matrix of daily returns |
| `efficient_frontier.png` | Markowitz efficient frontier with max-Sharpe portfolio |

### Next steps (open issues)

- Nelson-Siegel-Svensson (three-factor extension)
- Cubic / monotonic spline bootstrapping
- FRED / Treasury.gov data connectors
- Arbitrage-free / monotonicity unit tests on the fitted curve